# Raw preprocessing: sensitive steps to canonical groups

This tutorial explains the historical preprocessing contract. Production truth lives in
`analysis/raw/preprocess.py`; cells import and inspect it rather than copy it. Run top-to-bottom
with `study/requirements-analysis.txt` installed. The default tiny file is **synthetic**.
For real data, use the matching identity and generated count from a validated campaign.

## One row is one sensitive-gas Geant4 step

`Hits` stores every step, including zero energy deposits and particles excluded later.
`EventNumber` is local to one worker/run. `ParticleID` is an event-local track ID, **not a PDG code**;
`ParentID` identifies the parent track, which need not enter sensitive gas. Never join tracks on
ParticleID without EventNumber. `ParticleTag` is 0 e-, 1 e+, 2 gamma, 3 alpha, -1 otherwise.

`Nucleus` is the most recently tracked ion, not guaranteed ancestry. It can persist across events.
`ProcessType` is the track's **creator** process; it is not the process defining the current step.
Coordinates are pre-step world mm; `EnergyDeposit` is step MeV.

`GlobalTime_ns` is pre-step `GetGlobalTime()/ns`: event-relative Geant4 time.
Independent primary events have no absolute time relation. Legacy schema-0 raw files without
this branch yield `None` timing. New schema-1 files must carry timing and `OutputMetadata`.
`RunMetadata` identifies geometry/source; `RunAccounting` establishes the normalization denominator.


In [1]:
from pathlib import Path
import sys, inspect, tempfile, json, copy
import numpy as np
# Start Jupyter in the repository or any subdirectory (also works in archived source/).
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study/analyze.py").is_file())
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
try:
    from IPython.display import display, Markdown, Image
except ImportError:  # lightweight cell execution test; Jupyter itself supplies these
    display = print
    Markdown = str
from analysis.common.examples import IDENTITY, GEOMETRY
import uproot
from analysis.common.examples import write_example
from analysis.common.io import output_metadata
# Optional real worker file: supply its matching identity/accounting/geometry too.
INPUT_FILE = None
expected = IDENTITY
requested = 2
geometry = GEOMETRY
example_directory = tempfile.TemporaryDirectory(prefix="cygno-notebook-")
if INPUT_FILE is None:
    INPUT_FILE = Path(example_directory.name) / "synthetic-both.root"
    write_example(INPUT_FILE)
    print("SYNTHETIC SOFTWARE EXAMPLE — not a Monte Carlo rate estimate")
with uproot.open(INPUT_FILE) as file:
    print(output_metadata(file))


SYNTHETIC SOFTWARE EXAMPLE — not a Monte Carlo rate estimate
{'OutputFormat': 'both', 'OutputSchemaVersion': 1, 'TimingDefinition': 'pre-step global time / ns; Geant4 event-relative, not inter-event time', 'CompactProcessingVersion': 'event-boundaries-and-eof-v2'}


## Inspect the actual validated schema and representative steps

In [2]:
from analysis.raw.io import HITS_SCHEMA, LEGACY_HITS_SCHEMA, header, hit_chunks
from analysis.raw.preprocess import groups
from analysis.common.model import Group, GroupVolume
print(HITS_SCHEMA)
with uproot.open(INPUT_FILE) as file:
    accounting, hits = header(file, expected, requested)
    display(hits.arrays(entry_stop=8, library="np"))
print(inspect.getsource(Group.add))


{'EventNumber': 'int32', 'ParticleName': 'string', 'ParticleID': 'int32', 'ParticleTag': 'int32', 'ParentID': 'int32', 'x_hits': 'float64', 'y_hits': 'float64', 'z_hits': 'float64', 'EnergyDeposit': 'float64', 'VolumeNumber': 'int32', 'Nucleus': 'string', 'ProcessType': 'string', 'GlobalTime_ns': 'float64'}
{'EventNumber': array([0, 0, 0, 1, 1], dtype=int32), 'ParticleName': array(['e-', 'e-', 'e-', 'gamma', 'e-'], dtype=object), 'ParticleID': array([2, 2, 3, 1, 2], dtype=int32), 'ParticleTag': array([0, 0, 0, 2, 0], dtype=int32), 'ParentID': array([1, 1, 1, 0, 1], dtype=int32), 'x_hits': array([ 0., 99.,  0.,  0., 40.]), 'y_hits': array([0., 0., 0., 0., 0.]), 'z_hits': array([0., 0., 0., 0., 0.]), 'EnergyDeposit': array([0.   , 0.012, 0.003, 0.   , 0.4  ]), 'VolumeNumber': array([0, 0, 1, 0, 0], dtype=int32), 'Nucleus': array(['synthetic nucleus', 'synthetic nucleus', 'synthetic nucleus',
       'synthetic nucleus', 'synthetic nucleus'], dtype=object), 'ProcessType': array(['Radioacti

## Historical filtering and state transitions

Only e-, e+, and alpha steps enter grouping; gamma/neutrino steps are ignored **after** event-boundary
handling. Labels and creator process/nucleus drive the historical state, never track identity.
A matching creator/nucleus before ionization updates the final particle label. An ionization
transition (`eIoni` or `ionIoni`) otherwise joins and sets the flag. Another non-ionization
transition flushes. The first row of a new group intentionally leaves `ionization_seen=False`,
even if that row itself has an ionization creator. This inherited detail is essential.

End of event and EOF both flush the last nonempty group; uproot chunk boundaries have no effect.
The compact C++ implementation flushes explicitly at the sensitive detector's event end.


In [3]:
print(inspect.getsource(groups))

def groups(chunks, track_sink=None):
    group = None
    event_id = None
    index = 0
    tracks = {}
    fields = ('EventNumber', 'ParticleName', 'Nucleus', 'ProcessType', 'VolumeNumber',
              'EnergyDeposit', 'x_hits', 'y_hits', 'z_hits', 'ParticleID', 'ParticleTag', 'ParentID')
    def flush_tracks():
        if track_sink is not None:
            for key in sorted(tracks):
                track_sink(tracks[key])
        tracks.clear()
    for chunk in chunks:
        for i, row in enumerate(zip(*(chunk[f] for f in fields))):
            event, particle, nucleus, process, volume, energy, x, y, z, track, tag, parent = row
            if event != event_id:
                if group is not None:
                    yield group
                group = None
                flush_tracks()
                event_id, index = int(event), 0
            if track_sink is not None and int(track) not in tracks:
                tracks[int(track)] = TrackRecord(int(event), int(track), int(

## Per-volume accumulation: first position and first time travel together

Every admitted step adds energy in encounter order. The first encounter creates a volume entry,
even when its energy is zero. Subsequent steps change only energy. Both the saved position and
saved time come from that same first step: neither means minimum time nor first positive deposit.
Volumes keep first-encounter order. This matters because the historical cut reads the first
volume's first position. Timing never affects that cut.


In [4]:
with uproot.open(INPUT_FILE) as file:
    accounting, hits = header(file, expected, requested)
    tracks = []
    canonical = list(groups(hit_chunks(hits, requested, expected['layout'], step_size=2), tracks.append))
for group in canonical[:5]: display(group)
for track in tracks[:5]: display(track)


Group(event=0, particle='e-', nucleus='synthetic nucleus', process='RadioactiveDecay', group_index=0, volumes={0: GroupVolume(energy=0.012, x=0.0, y=0.0, z=0.0, first_hit_time_ns=17.0), 1: GroupVolume(energy=0.003, x=0.0, y=0.0, z=0.0, first_hit_time_ns=23.0)}, step_count=3, track_ids={2, 3})
Group(event=1, particle='e-', nucleus='synthetic nucleus', process='RadioactiveDecay', group_index=0, volumes={0: GroupVolume(energy=0.4, x=40.0, y=0.0, z=0.0, first_hit_time_ns=12.0)}, step_count=1, track_ids={2})
TrackRecord(event=0, particle_id=2, particle_tag=0, parent_id=1, particle='e-', nucleus='synthetic nucleus', process='RadioactiveDecay', group_indices=[0])
TrackRecord(event=0, particle_id=3, particle_tag=0, parent_id=1, particle='e-', nucleus='synthetic nucleus', process='eIoni', group_indices=[0])
TrackRecord(event=1, particle_id=1, particle_tag=2, parent_id=0, particle='gamma', nucleus='synthetic nucleus', process='RadioactiveDecay', group_indices=[])
TrackRecord(event=1, particle_id

## Checks and handoff

The synthetic first hit has zero energy, x=0 and t=17 ns; a later step has x=99 and t=25 ns.
Only its energy is added. Use the canonical Group objects directly in shared analysis.
A deliberately different reconstruction belongs in a separately versioned study.


In [5]:
with uproot.open(INPUT_FILE) as file:
    _, hits = header(file, expected, requested)
    assert canonical == list(groups(hit_chunks(hits, requested, expected['layout'], step_size=1)))
from analysis.common.spectra import accumulate
counts, histogram, n = accumulate(canonical, geometry)
assert np.all(counts[:, 0] == histogram.sum(axis=1))
print("Chunk parity and group bookkeeping passed", n)


Chunk parity and group bookkeeping passed 2
